In [50]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import os
import requests
from dotenv import load_dotenv
load_dotenv()
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [51]:
@tool
def getConversionFactor(base_currency:str,target_currency:str)-> float:
    """This function fetches the currency conversion factor of a given base currency and target currency"""
    url=f"https://v6.exchangerate-api.com/v6/{os.getenv("API_KEY")}/pair/{base_currency}/{target_currency}"
    response=requests.get(url)
    return response.json()

In [52]:
@tool
def convert(base_curr_value:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_curr_value * conversion_rate

In [53]:
getConversionFactor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1792195202,
 'time_last_update_utc': 'Sat, 17 Oct 2026 00:00:02 +0000',
 'time_next_update_unix': 1792281602,
 'time_next_update_utc': 'Sun, 18 Oct 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.112}

In [54]:
convert.invoke({'base_curr_value':10,'conversion_rate':95.112})

951.1199999999999

In [55]:
llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY_3")
)

In [56]:
llm_with_tools=llm.bind_tools([getConversionFactor,convert])

In [57]:
messages=[HumanMessage("WHat is the conversion factor between USD and NPR and based on that can you convert 12 USD to NPR ?")]

In [58]:
ai_message=llm_with_tools.invoke(messages)

In [59]:
messages.append(ai_message)

In [60]:
ai_message.tool_calls

[{'name': 'getConversionFactor',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'call_644927',
  'type': 'tool_call'}]

In [61]:
import json
for tool_call in ai_message.tool_calls:
    if tool_call['name']=='getConversionFactor':
        tool_msg1=getConversionFactor.invoke(tool_call)
        conversion_rate=json.loads(tool_msg1.content)['conversion_rate']
        messages.append(tool_msg1)
    if tool_call['name']=='convert':
        tool_call['args']['conversion_rate']=conversion_rate
        tool_msg2=convert.invoke(tool_call)
        messages.append(tool_msg2)


In [62]:
llm_with_tools.invoke(messages)

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_curr_value": 12}'}, '__gemini_function_call_thought_signatures__': {'call_2385601': 'EmAKXgFpFH0THY+OeBRLWwWQXt/kZqdfRFI8nDFsZdl0fBt104ZJy4qU/+yc/jcsOKv4G1WvpZUkSYm03Z0sldzNO1rmrphRGqVnFv3r5+81zNwL+kWH7QpUeTSP1HCRA0E='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14976-612b-7c61-9dd0-6d19066c5662-0', tool_calls=[{'name': 'convert', 'args': {'base_curr_value': 12}, 'id': 'call_2385601', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 379, 'output_tokens': 17, 'total_tokens': 396, 'input_token_details': {'cache_read': 0}})